# VDS on Coiled: HTTP virtual chunks, icechunk store in S3

This notebook shows the split you usually want in production:

- **granules are read over HTTPS** (the default `access="indirect"`), so the
  virtual chunks point at `https://…` URLs and the workers need no AWS
  credentials;
- **the icechunk store itself lives in your own S3 bucket**
  (`s3://its-live-data/test-space/vds/<DATASET>`), so it survives the cluster
  and other people can open it.

The per-granule opens run on the Coiled workers; the combine + write + commit
run in the job thread on your client. That is why only the client needs write
access to the bucket.

> Replace `DATASET`, the bucket credentials and (if needed) the region with
> your own. The bucket must support conditional writes (plain AWS S3 does).

## 1. Log in and search

The EDL token is used for two different things: the HTTPS granule/DMR++ reads
and, later, the `Authorization` header injected into NASA HTTP containers when
reading the store back.

In [ ]:
import os

import coiled
import earthaccess as ea
import icechunk

ea.login(strategy="environment")  # or "netrc" / "interactive"

DATASET = "MUR-SST"
BUCKET = "its-live-data"
PREFIX = f"test-space/vds/{DATASET}"
STORE = f"s3://{BUCKET}/{PREFIX}"
REGION = "us-west-2"  # the bucket's region, not the data's

granules = ea.search_data(
    short_name="MUR-JPL-L4-GLOB-v4.1",
    temporal=("2013-01-01", "2013-01-10"),
)
len(granules)

## 2. Coiled cluster

Workers need `earthaccess` (the `plugins` branch until v2 is released), the
plugin, and the virtual-dataset stack. `coiled.Cluster(packages=[...])` builds
the environment from those specs.

In [ ]:
cluster = coiled.Cluster(
    name="vds-http-to-s3",
    n_workers=8,
    region=REGION,
    packages=[
        "git+https://github.com/earthaccess-dev/earthaccess.git@plugins",
        "git+https://github.com/earthaccess-dev/earthaccess-plugin-vds.git",
        "virtualizarr>=2.8",
        "h5py>=3.14",
        "icechunk>=2.3",
        "obstore>=0.12",
    ],
)
client = cluster.get_client()
client

## 3. Give the workers the EDL token (not the AWS keys)

`distribute_credentials` uses Coiled's private environment channel: values are
never put in the task payload, and it returns only the variable names. The
workers do not need S3 credentials, because they only read HTTPS.

In [ ]:
ea.distribute_credentials(cluster=cluster)

## 4. Create the store in S3

### Option A — a plain `s3://` target (credentials from the environment)

The plugin resolves `s3://` targets with `icechunk.S3Credentials.FromEnv()`, so
export the usual AWS variables first. `AWS_DEFAULT_REGION` matters: it is how
icechunk knows where the bucket is.

In [ ]:
os.environ["AWS_ACCESS_KEY_ID"] = "..."
os.environ["AWS_SECRET_ACCESS_KEY"] = "..."
os.environ["AWS_SESSION_TOKEN"] = "..."  # if you have temporary creds
os.environ["AWS_DEFAULT_REGION"] = REGION

job = ea.plugins.vds.request(
    granules,
    store=STORE,  # icechunk repository created under this prefix
    concat_dim="time",
    access="indirect",  # HTTPS virtual chunks (the default)
    client=client,
    show_progress=True,
)
job

### Option B — a preconfigured `icechunk.Storage` (explicit credentials/region)

Use this when the credentials are not in the environment, the bucket is not in
the default region, or you point at an S3-compatible endpoint. A `Storage`
carries no readable path, so pass `href=` for the returned `Asset`.

In [ ]:
storage = icechunk.Storage.new_s3_object_store(
    icechunk.S3Options(
        region=REGION
    ),  # endpoint_url=..., allow_http=... for non-AWS S3
    BUCKET,
    PREFIX,
    credentials=icechunk.S3StaticCredentials(
        access_key_id="...",
        secret_access_key="...",
        session_token=None,  # or a temporary token
    ),
)

job = ea.plugins.vds.request(
    granules,
    store=storage,  # an icechunk.Storage, not a path
    href=STORE,  # required: the Asset needs a readable location
    concat_dim="time",
    client=client,
    show_progress=True,
)
job

### Optional — fix the manifest layout at creation

`repository_config=` is applied when the repository is created (manifest
splitting, inline-chunk threshold, compression, …). It is ignored with a
warning on append, where the stored configuration wins.

In [ ]:
config = icechunk.RepositoryConfig.default()
config.manifest = icechunk.ManifestConfig(
    splitting=icechunk.ManifestSplittingConfig.from_dict(
        {
            icechunk.ManifestSplitCondition.name_matches("analysed_sst"): {
                icechunk.ManifestSplitDimCondition.DimensionName("time"): 64,
            },
        }
    )
)

job = ea.plugins.vds.request(
    granules,
    store=storage,
    href=STORE,
    concat_dim="time",
    repository_config=config,
    client=client,
)
job

## 5. Wait for the job and keep the asset

`request()` returns immediately. `job.info()` reports the phase
(`virtualize` → `combine` → `write`) and the per-granule counters; `results()`
blocks and returns the single `Asset`.

In [ ]:
asset = job.results()[0]
asset

The store now lives at
`s3://its-live-data/test-space/vds/MUR-SST` and references the NASA HTTPS
granule chunks. The virtual chunk containers were written into the repository
config, one per `https://host/` found in the chunk references.

## 6. Read it back (from any machine)

Reading needs read access to the bucket **and** an EDL login: `open_virtual()`
authorizes the stored containers and injects the bearer token into NASA HTTP
containers automatically. Pass `storage_options` when the reader has no AWS
environment credentials.

In [ ]:
ea.login(strategy="environment")

ds = ea.open_virtual(
    asset.href,
    storage_options={
        "region": REGION,
        "access_key_id": "...",
        "secret_access_key": "...",
        # "session_token": "...",
    },
)
ds

If you need different credentials for a chunk host, override the mapping — keys
match the container `url_prefix` exactly (authorizing `s3://bucket/` does **not**
authorize `s3://bucket/sub/`). `HttpAccess`/`LocalFileSystemAccess` are no-auth
sentinels; S3 needs real credentials (`S3Credentials.FromEnv()`,
`s3_anonymous_credentials()`, …).

In [ ]:
ds = ea.open_virtual(
    asset.href,
    storage_options={
        "region": REGION,
        "access_key_id": "...",
        "secret_access_key": "...",
    },
    authorize_virtual_chunk_access={
        "https://podaac-ops-cumulus-protected.s3.us-west-2.amazonaws.com/": (
            icechunk.credentials.HttpAccess
        ),
    },
)
ds

## 7. Append more granules later

Same call, same store: the repository already exists, so `concat_dim` doubles as
`append_dim` and icechunk appends the new time steps without rewriting the old
ones.

In [ ]:
more = ea.search_data(
    short_name="MUR-JPL-L4-GLOB-v4.1",
    temporal=("2013-01-11", "2013-01-20"),
)

append_job = ea.plugins.vds.request(
    more,
    store=storage,
    href=STORE,
    concat_dim="time",
    client=client,
)
append_job.results()[0]

## Notes and gotchas

- **Write side is the client.** Only the client needs write access to the
  bucket; workers just open granules over HTTPS. If you switch to
  `access="direct"`, workers would need the NASA S3 credentials too.
- **Region.** Check the bucket's region (`its-live-data` is in `us-west-2`). A
  wrong region shows up as a redirect/`301` error on the first write; Option B
  with `S3Options(region=...)` is the reliable fix.
- **S3-compatible endpoints.** Use
  `icechunk.S3Options(region=..., endpoint_url=..., allow_http=True, force_path_style=True)`
  and the `Storage` form (Option B).
- **Credentials are never serialized.** They live in the client process (and,
  for the EDL token, in the workers' private environment), never in the job
  payload or the task graph.
- **Appends** need the stored config to still authorize every host in the new
  batch. The plugin merges missing containers and persists them, or pass
  `virtual_chunk_containers=` explicitly.

In [ ]:
client.close()
cluster.close()